In [ ]:
# Imports
import time

import matplotlib.pyplot as plt
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import sim_function

figures_dir = Path("figures")
figures_dir.mkdir(exist_ok=True)


In [ ]:
# Settings
nsim = 100000
ngenest = 3  # Rho uses cases 0–2; growth uses generation counts 0–3
nubar_range = (0.5, 1.5)
theta_range = (0.1, 1)  # Dispersion parameter k
delta_range = (0, 0.3)
q = np.round(np.linspace(0.5, 0.9, 9), 2)
q_plot = (0.5, 0.7, 0.9)

# Transmission parameters are resampled for each outbreak
para = {
    'nubar': 1,
    'theta': 1,
    'delta': 0,
    'boundary': 1000,
    'ngen': 128,  # Length of returned generation summaries
    'ninit': 1,
}

rmax = 5


In [ ]:
# Analysis helpers
def rho_column(window, quantile):
    return f'rho_{window}_q_{quantile:.2f}'


def estimate_rho(data, quantiles, generation_count=None):
    if generation_count is not None:
        data = data[data['gen'] < generation_count]

    edges = data[data['parent'].notna()]
    offspring = data['on']
    ratios = dict.fromkeys(quantiles, np.nan)
    if edges.empty:
        return ratios

    # Empirical rank formula applied to sampled offspring counts
    child_ranks = rankdata(offspring.loc[edges.index], method='average') / len(edges)
    parent_ranks = rankdata(
        offspring.loc[edges['parent'].astype(int)], method='average'
    ) / len(edges)
    for quantile in quantiles:
        top = child_ranks >= quantile
        if top.any() and (~top).any():
            ratios[quantile] = np.mean(parent_ranks[top]) / np.mean(parent_ranks[~top])
    return ratios


def estimate_predictors(out, generation_count, quantiles, threshold):
    generation_sizes = out['gen_size'][:generation_count + 1]
    if len(generation_sizes) != generation_count + 1:
        return None
    if not np.all(np.isfinite(generation_sizes) & (generation_sizes > 0)):
        return None

    observed_size = int(generation_sizes.sum())
    if observed_size >= threshold:
        return None

    generations = np.arange(generation_count + 1)
    slope, _ = np.polyfit(generations, np.log(generation_sizes), 1)
    predictors = {
        'slope': slope,
        'observed_size': observed_size,
        'final_size': out['final_size'],
        'reached_threshold': out['final_size'] >= threshold,
    }
    for window, limit in (('early', generation_count), ('full', None)):
        ratios = estimate_rho(out['data'], quantiles, limit)
        predictors.update({
            rho_column(window, quantile): ratio for quantile, ratio in ratios.items()
        })
    return predictors


def growth_factors(slopes, max_growth):
    growth = np.exp(slopes).copy()
    for cutoff in (1.0, 2.0, float(max_growth)):
        near_cutoff = np.isclose(growth, cutoff, rtol=0, atol=1e-12)
        growth.loc[near_cutoff] = cutoff
    return growth


def filter_outbreaks(data, max_growth, quantile=None):
    selected = data[growth_factors(data['slope'], max_growth) <= max_growth]
    if quantile is not None:
        columns = [rho_column(window, quantile) for window in ('early', 'full')]
        selected = selected[np.isfinite(selected[columns]).all(axis=1)]
    return selected.copy()


def plot_epidemic_proportions(
    data, stratify_by, label_prefix, max_growth, threshold, quantile=None, ax=None
):
    selected = filter_outbreaks(data, max_growth, quantile)
    bins = np.linspace(selected['slope'].min(), selected['slope'].max(), 8)
    selected['slope_bin'] = pd.cut(selected['slope'], bins, include_lowest=True)
    # Tied rho values can prevent four distinct quartile boundaries
    selected['quartile'] = pd.qcut(selected[stratify_by], 4, labels=False, duplicates='drop')

    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots()
    else:
        fig = ax.figure
    r_theory = np.linspace(1.0, max_growth, 200)
    ax.plot(r_theory, 1 - 1 / r_theory, color='grey', linestyle='--', label='theory')

    colors = plt.cm.plasma(np.linspace(0, 1, selected['quartile'].nunique() + 1))[:-1]
    for (quartile, subset), color in zip(selected.groupby('quartile'), colors):
        proportions = subset.groupby('slope_bin', observed=True)['reached_threshold'].mean()
        r_centers = np.exp([interval.mid for interval in proportions.index])
        ax.plot(
            r_centers, proportions, marker='o', color=color,
            label=f'{label_prefix}{quartile + 1}',
        )

    if quantile is not None:
        ax.set_title(f'q = {quantile:g}')
    ax.set(xlabel='R', ylabel=f'p(reach {threshold:,} cases)')
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], labels[::-1], title='')
    if standalone:
        plt.show()
    return fig, ax


def compare_auc(data, quantile):
    outcome = data['reached_threshold']
    folds = list(StratifiedKFold(n_splits=5).split(data, outcome))

    classifier = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    auc_base = cross_val_score(
        classifier, data[['slope']], outcome, cv=folds, scoring='roc_auc'
    )
    mean_auc_base = auc_base.mean()
    predictor_sets = {
        'early': ['slope', rho_column('early', quantile)],
        'full': ['slope', rho_column('full', quantile)],
        'oracle': ['slope', 'delta'],
    }
    results = {}
    for model, columns in predictor_sets.items():
        auc_model = cross_val_score(
            classifier, data[columns], outcome, cv=folds, scoring='roc_auc',
        )
        delta_auc = auc_model.mean() - mean_auc_base
        results[model] = {
            'n': len(data),
            'auc_base': auc_base,
            'auc_model': auc_model,
            'delta_auc': delta_auc,
            'auc_gap_closed': delta_auc / (1 - mean_auc_base),
        }
    return results


In [ ]:
# Simulate outbreaks and estimate predictors
records = []
start_time = time.time()

for i in range(nsim):
    if (i + 1) % 200 == 0 or (i + 1) == nsim:
        elapsed = time.time() - start_time
        rate = (i + 1) / max(elapsed, 1e-9)
        remaining = (nsim - i - 1) / max(rate, 1e-9)
        print(
            f'{i + 1}/{nsim}  |  {rate:.1f} it/s  |  '
            f'Elapsed: {elapsed:.1f} sec | ETA ~{remaining:.1f} sec',
            end='\r',
        )

    para['nubar'] = np.random.uniform(*nubar_range)
    para['delta'] = np.random.uniform(*delta_range)
    para['theta'] = np.random.uniform(*theta_range)
    out = sim_function.sim(para)
    predictors = estimate_predictors(out, ngenest, q, para['boundary'])
    if predictors is None:
        continue

    records.append({
        'simulation': i,
        'nubar': para['nubar'],
        'theta': para['theta'],
        'delta': para['delta'],
        **predictors,
    })

outbreak_data = pd.DataFrame.from_records(records)
print(f'\nEligible at generation {ngenest}: {len(outbreak_data)} of {nsim} simulations')


In [ ]:
# Derived quantities and growth distribution
outbreak_data['delta_star'] = (
    (1 - outbreak_data['nubar'])
    / (1 - outbreak_data['nubar'] + 1 / outbreak_data['theta'])
)
outbreak_data['epsilon'] = outbreak_data['delta'] - outbreak_data['delta_star']
outbreak_data['r_est'] = growth_factors(outbreak_data['slope'], rmax)
outbreak_data['r_est'].describe()


In [ ]:
# Epidemic proportions by epsilon
plot_epidemic_proportions(outbreak_data, 'epsilon', 'ε = q', rmax, para['boundary']);


In [ ]:
# Epidemic proportions by autocorrelation
fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
for col, quantile in enumerate(q_plot):
    for row, window in enumerate(('early', 'full')):
        plot_epidemic_proportions(
            outbreak_data, rho_column(window, quantile),
            f'{window.capitalize()} ρ quartile ', rmax, para['boundary'], quantile,
            ax=axes[row, col],
        )
plt.show()


In [ ]:
# Classifier comparison
prediction_data = {}
prediction_subsets = {}
auc_results = {}
for quantile in q:
    selected = filter_outbreaks(outbreak_data, rmax, quantile)
    prediction_data[quantile] = selected
    print(
        f'q = {quantile:g}: kept {len(selected)} of {len(outbreak_data)} eligible '
        f'simulations with R <= {rmax} and both rho estimates defined'
    )

    prediction_growth = growth_factors(selected['slope'], rmax)
    subsets = {
        'qualifying': selected,
        'low': selected[prediction_growth <= 2],
        'mid': selected[prediction_growth > 2],
    }
    prediction_subsets[quantile] = subsets
    for subset, data in subsets.items():
        for model, result in compare_auc(data, quantile).items():
            auc_results[(quantile, model, subset)] = result

auc_summary = pd.DataFrame.from_dict(
    {
        key: {
            'n': result['n'],
            'auc_base': result['auc_base'].mean(),
            'auc_model': result['auc_model'].mean(),
            'delta_auc': result['delta_auc'],
            'auc_gap_closed': result['auc_gap_closed'],
        }
        for key, result in auc_results.items()
    },
    orient='index',
)
auc_summary.index = pd.MultiIndex.from_tuples(
    auc_summary.index, names=['q', 'model', 'subset']
)

# Match all models within each q; qualifying counts can vary by q
comparison_table = auc_summary.xs('early', level='model')[['n', 'auc_base']].copy()
comparison_table['n'] = comparison_table['n'].astype(int)
model_aucs = auc_summary['auc_model'].unstack('model').rename(columns={
    'early': 'auc_early', 'full': 'auc_full', 'oracle': 'auc_oracle',
})
comparison_table = comparison_table.join(model_aucs[['auc_early', 'auc_full', 'auc_oracle']])
comparison_table.loc[list(q_plot)]


In [ ]:
# LaTeX table
latex_table = comparison_table.loc[list(q_plot)].copy()
latex_table.index = pd.MultiIndex.from_tuples(
    [(f'{quantile:g}', subset) for quantile, subset in latex_table.index],
    names=[r'$q$', 'Subset'],
)
latex_table = latex_table.rename(columns={
    'n': r'$n$',
    'auc_base': r'\shortstack{AUC\\($R$)}',
    'auc_early': r'\shortstack{AUC\\($R$ + early $\rho$)}',
    'auc_full': r'\shortstack{AUC\\($R$ + full $\rho$)}',
    'auc_oracle': r'\shortstack{AUC\\($R$ + true $\delta$)}',
})
caption = (
    f'Classification performance among qualifying outbreaks from {nsim:,} simulations. '
    + r"""Entries are mean five-fold cross-validated areas under the receiver-operating characteristic curve (AUC); $n$ is the number of trees in each row, and $q$ is the rank threshold used to estimate $\rho$.
Both early predictors use observations through generation 3: growth $R$ is the exponentiated slope of log case counts in generations 0--3, and early $\rho$ uses offspring counts of cases in generations 0--2.
Full-tree $\rho$ uses all cases included before simulation stops and gives a retrospective comparison.
Qualifying outbreaks reach generation 3 without exceeding 1000 cases, """
    + rf'$R \leq {rmax:g}$, and both $\rho$ estimates defined. '
    + r"""The low and mid subsets partition qualifying outbreaks into $R \leq 2$ and """
    + rf'$2 < R \leq {rmax:g}$, respectively. '
    + r"""Classifiers use log $R$ alone, with early or full-tree $\rho$, or with the true simulated $\delta$ (oracle), using matched outbreaks and cross-validation folds within each $q$.
Predictors are standardized using only the training data in each fold.
The oracle assumes $\delta$ is known without estimation error.
Outcomes indicate reaching 1000 cases after the early observation window."""
)
print(r"""\begin{table}[ht]
\centering
""" + latex_table.to_latex(
    column_format='llrcccc',
    float_format='%.4f',
    formatters={r'$n$': lambda value: f'{int(value):,}'},
    escape=False, sparsify=True, multirow=False,
) + r"""\caption{
""" + caption + r"""
}\label{tab:auc_results}
\end{table}
""")


In [ ]:
# True delta versus estimated rho across rank thresholds
from scipy.stats import t as student_t

fig, axes = plt.subplots(
    len(q_plot), 2, figsize=(9, 2.8 * len(q_plot)),
    sharex=True, sharey=True, squeeze=False, layout='constrained',
)
for row, scatter_q in enumerate(q_plot):
    scatter_data = filter_outbreaks(outbreak_data, rmax, scatter_q)
    delta_values = scatter_data['delta']
    for ax, window in zip(axes[row], ('early', 'full')):
        rho_values = scatter_data[rho_column(window, scatter_q)]
        size_column = 'observed_size' if window == 'early' else 'final_size'
        # Shared area scale; early size counts cases through the observation cutoff.
        point_areas = np.maximum(4.0, 20.0 * scatter_data[size_column] / para['boundary'])
        ax.scatter(
            delta_values, rho_values,
            s=point_areas, alpha=1, marker='o',
            facecolors='none', edgecolors='#3B7A9C', linewidths=0.35,
        )
        ax.set(xlim=delta_range, ylim=(0, 2))
        ax.set_xticks(np.linspace(*delta_range, 4))
        ax.set_yticks(np.arange(0, 2.1, 0.5))
        ax.spines[['top', 'right']].set_visible(False)
        ax.spines[['bottom', 'left']].set(color='0.5', linewidth=0.6)
        ax.tick_params(labelsize=12, length=3, width=0.6, color='0.5')

        panel_label = chr(ord('a') + row + (0 if window == 'early' else len(q_plot)))
        ax.text(
            0.025, 0.97, panel_label, transform=ax.transAxes,
            ha='left', va='top', fontsize=18, fontweight='bold', zorder=4,
            bbox=dict(facecolor='white', edgecolor='none', pad=1, alpha=0.9),
        )
        if window == 'early':
            ax.text(
                0.98, 0.97, rf'$q = {scatter_q:g}$' + f'  (n = {len(scatter_data):,})',
                transform=ax.transAxes, ha='right', va='top', fontsize=10, zorder=4,
                bbox=dict(facecolor='white', edgecolor='none', pad=1, alpha=0.9),
            )
        above_limit = rho_values > 2
        if above_limit.any():
            percent_above = 100 * above_limit.mean()
            percent_label = '<0.1%' if percent_above < 0.1 else f'{percent_above:.1f}%'
            ax.text(
                0.98, 0.025, f'{percent_label} above 2 (max {rho_values.max():.2f})',
                transform=ax.transAxes, ha='right', va='bottom', fontsize=8,
                color='0.35', zorder=4,
                bbox=dict(facecolor='white', edgecolor='none', pad=1, alpha=0.9),
            )

        # polyfit weights multiply residuals, so sqrt(size) weights squared errors by size.
        (fit_slope, fit_intercept), fit_cov = np.polyfit(
            delta_values, rho_values, 1,
            w=np.sqrt(scatter_data[size_column]), cov=True,
        )
        fit_delta = np.linspace(delta_values.min(), delta_values.max(), 100)
        ax.plot(
            fit_delta, fit_intercept + fit_slope * fit_delta,
            color='#D66A28', linewidth=1.8, zorder=3,
        )
        # Two-sided t-test of slope = 0 for the same tree-size-weighted fit.
        slope_se = np.sqrt(fit_cov[0, 0])
        slope_p = 2 * student_t.sf(abs(fit_slope / slope_se), df=len(scatter_data) - 2)
        if slope_p < 1e-9:
            p_label = r'$p < 10^{-9}$'
        else:
            p_coefficient, p_exponent = f'{slope_p:.2e}'.split('e')
            p_label = rf'$p = {p_coefficient} \times 10^{{{int(p_exponent)}}}$'
        ax.text(
            0.025, 0.025,
            f'Intercept = {fit_intercept:.3f}\nSlope = {fit_slope:.3f}\n{p_label}',
            transform=ax.transAxes, ha='left', va='bottom', fontsize=11, zorder=4,
            bbox=dict(facecolor='white', edgecolor='none', pad=1, alpha=0.9),
        )
        if row == 0:
            ax.set_title('Early tree' if window == 'early' else 'Full tree', fontsize=12, pad=8)
fig.supxlabel(r'Transmission autocorrelation, $\delta$', fontsize=14)
fig.supylabel(r'Infector rank ratio, $\rho$', fontsize=14)
plt.show()


In [ ]:
# Save a compact PDF: rasterize the points, retaining vector text and fit lines.
output_path = figures_dir / "delta_rho.pdf"
for ax in fig.axes:
    for collection in ax.collections:
        collection.set_rasterized(True)
fig.savefig(output_path, format="pdf", dpi=600, bbox_inches="tight")
